<a href="https://colab.research.google.com/github/ishwarraja/SOAI/blob/main/ERAv4/S12/S12_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Configuration for 124M Model ---
@dataclass
class GPTConfig:
    block_size: int = 1024  # Context length (T)
    vocab_size: int = 50257 # Will be updated dynamically based on input.txt
    n_layer: int = 12       # L (Layers) - Set for GPT-2 Small (124M)
    n_head: int = 12        # H (Heads) - Set for GPT-2 Small (124M)
    n_embd: int = 768       # D (Embedding dimension) - Set for GPT-2 Small (124M)

# --- Transformer Components (as provided in initial file snippet) ---

class CausalSelfAttention(nn.Module):
    """A minimal Causal Self-Attention block."""
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # Scale init to 1.0/sqrt(2L) for residual connection stability
        self.c_proj.NANGPT_SCALE_INIT = 1.0 / math.sqrt(2.0 * config.n_layer)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # Causal mask (tril) is registered as buffer
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                            .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # Causal self-attention; (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)

        y = att @ v # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs

        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    """A minimal Multi-Layer Perceptron block."""
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        # Scale init to 1.0/sqrt(2L) for residual connection stability
        self.c_proj.NANGPT_SCALE_INIT = 1.0 / math.sqrt(2.0 * config.n_layer)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    """A minimal Transformer Block consisting of Attention and MLP."""
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    """The full GPT model composed of Blocks."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying

        self.apply(self._init_weights)
        print(f"Number of parameters: {self.get_num_params()/1e6:.2f} Million")

    def get_num_params(self, non_embedding=True):
        """Return the number of parameters in the model."""
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        """Custom weight initialization."""
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANGPT_SCALE_INIT'):
                std *= module.NANGPT_SCALE_INIT
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        device = idx.device
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"

        pos = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(0) # shape (1, T)

        tok_emb = self.transformer.wte(idx) # token embeddings (B, T, D)
        pos_emb = self.transformer.wpe(pos) # position embeddings (1, T, D)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Flatten B, T, V to (B*T), V for cross-entropy loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss

# --- Data Loading and Tokenization (Character-level for Simplicity) ---
class DataLoaderLite:
    def __init__(self, input_text, B, T):
        self.B = B # batch size
        self.T = T # context length

        # Character-level tokenizer
        chars = sorted(list(set(input_text)))
        self.vocab_size = len(chars)
        self.stoi = {ch:i for i,ch in enumerate(chars)}
        self.itos = {i:ch for i,ch in enumerate(chars)}
        self.encode = lambda s: [self.stoi[c] for c in s]
        self.decode = lambda l: ''.join([self.itos[i] for i in l])

        # Convert text to tokens
        self.tokens = torch.tensor(self.encode(input_text), dtype=torch.long)
        print(f"Loaded data with {len(self.tokens)} tokens, vocabulary size: {self.vocab_size}")

        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]

        if buf.size(0) < B * T + 1:
            # Wrap around when end of data is reached
            remainder = B * T + 1 - buf.size(0)
            buf = torch.cat((buf, self.tokens[:remainder]))
            self.current_position = remainder # Restart from the beginning
        else:
            self.current_position += B * T

        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        return x, y

# --- Main Execution and Training Loop ---

def generate(model, start_str, max_new_tokens, temperature=1.0, top_k=50):
    """Generates new text given a starting string."""
    model.eval()
    B, T = 1, model.config.block_size

    # Encode the starting string
    start_ids = model.data_loader.encode(start_str)
    x = (torch.tensor(start_ids, dtype=torch.long, device=model.lm_head.weight.device)[None, ...])

    # Truncate if the start string is too long
    x = x[:, -T:]

    for _ in range(max_new_tokens):
        # crop context if necessary
        x_cond = x if x.size(1) <= T else x[:, -T:]

        with torch.no_grad():
            logits, _ = model(x_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally apply top_k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            # sample from the distribution
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # append sampled index to the running sequence
        x = torch.cat((x, idx_next), dim=1)

        # stop if we predict the newline token (often a good proxy for end of generation)
        if idx_next.item() == model.data_loader.stoi.get('\n'):
             break

    return model.data_loader.decode(x[0].tolist())

def main():
    # 1. Setup Environment and Load Data

    # Check for GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Set seeds for reproducibility
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)

    try:
        # NOTE: Assumes input.txt is available in the execution environment
        with open('input.txt', 'r') as f:
            input_text = f.read()
    except FileNotFoundError:
        print("ERROR: 'input.txt' not found. Please ensure the file is downloaded to the execution directory.")
        return

    # 2. Initialize Data Loader and Model
    B, T = 4, 1024 # Batch size 4, Context length 1024 (Adjust B down if out-of-memory on Colab T4)
    train_loader = DataLoaderLite(input_text, B=B, T=T)

    # Update config with the dynamically determined vocab size
    config = GPTConfig(vocab_size=train_loader.vocab_size, block_size=T)
    model = GPT(config).to(device)
    model.data_loader = train_loader # Attach data loader for easy access in generate function

    # 3. Training Setup
    max_steps = 10000  # A high number of steps is necessary for the ambitious loss target (< 0.1)
    eval_interval = 100
    log_interval = 10

    optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, betas=(0.9, 0.95), weight_decay=0.1)

    # 4. Training Loop
    start_time = time.time()

    for step in range(max_steps):
        # Set model to training mode
        model.train()

        # Fetch batch and move to device
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)

        # Forward pass and backpropagation
        optimizer.zero_grad()
        logits, loss = model(x, y)
        loss.backward()

        # Simple gradient clipping to stabilize training
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        # Logging and Evaluation
        current_loss = loss.item()

        if step % log_interval == 0:
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps} | Loss: {current_loss:.6f} | Time: {elapsed_time:.2f}s")

            # Check for early stopping condition (Loss < 0.099999)
            if current_loss < 0.099999:
                print("\n" + "="*50)
                print(f"SUCCESS: Loss target reached at Step {step+1}!")
                print("="*50 + "\n")

                # Final save and sample generation
                torch.save(model.state_dict(), f'gpt_124m_final_loss_{current_loss:.4f}.pt')
                break

        if step % eval_interval == 0 and step > 0:
            print("-" * 50)
            print("--- Generating Sample Output ---")

            # Generate sample text
            prompt = "First Citizen:"
            generated_text = generate(model, prompt, max_new_tokens=200, temperature=0.9)

            print(f"\n[Prompt]: {prompt}\n")
            print(f"[Generated Text]:\n{generated_text}")
            print("-" * 50)

            # Save checkpoint
            torch.save(model.state_dict(), f'checkpoint_step_{step}.pt')

    print("\nTraining complete.")
    if current_loss >= 0.099999:
        print(f"Note: Final loss of {current_loss:.6f} did not meet the target of < 0.099999. Try running for more steps or adjusting hyperparameters.")

if __name__ == '__main__':
    main()

Using device: cuda
Loaded data with 1115394 tokens, vocabulary size: 65
Number of parameters: 85.11 Million
Step 1/10000 | Loss: 4.401192 | Time: 1.61s
Step 11/10000 | Loss: 3.789780 | Time: 10.17s
Step 21/10000 | Loss: 3.321978 | Time: 18.81s
Step 31/10000 | Loss: 3.256943 | Time: 27.52s
Step 41/10000 | Loss: 3.367169 | Time: 36.34s
Step 51/10000 | Loss: 3.089418 | Time: 45.26s
Step 61/10000 | Loss: 2.756719 | Time: 54.28s
Step 71/10000 | Loss: 2.717390 | Time: 63.36s
Step 81/10000 | Loss: 2.656336 | Time: 72.55s
Step 91/10000 | Loss: 2.633832 | Time: 81.82s
Step 101/10000 | Loss: 2.526963 | Time: 91.16s
--------------------------------------------------
--- Generating Sample Output ---

[Prompt]: First Citizen:

[Generated Text]:
First Citizen:

--------------------------------------------------
Step 111/10000 | Loss: 2.568762 | Time: 106.89s
Step 121/10000 | Loss: 2.546995 | Time: 116.02s
Step 131/10000 | Loss: 2.557807 | Time: 125.19s
Step 141/10000 | Loss: 2.498006 | Time: 134.35s